In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "235_build_silver_dim_project_classification.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.dim_project"
TARGET_TABLE = f"{catalog}.silver.dim_project_classification"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build dim_project_classification
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH source_values AS (
        SELECT DISTINCT
            TRIM(project_classification) AS project_classification
        FROM {SOURCE_TABLE}
    )

    , normalized AS (
        SELECT
            CASE
                WHEN project_classification IS NULL
                  OR project_classification = ''
                THEN 'UNKNOWN'
                ELSE project_classification
            END AS project_classification
        FROM source_values
    )

    , distinct_values AS (
        SELECT DISTINCT
            project_classification
        FROM normalized
    )

    , ordered_values AS (
        SELECT
            project_classification,
            ROW_NUMBER() OVER (
                ORDER BY project_classification
            ) AS seq_nbr
        FROM distinct_values
        WHERE project_classification <> 'UNKNOWN'
    )

    SELECT
        CAST(-1 AS INT) AS project_classification_key,
        CAST('UNKNOWN' AS STRING) AS project_classification

    UNION ALL

    SELECT
        CAST(seq_nbr AS INT) AS project_classification_key,
        project_classification
    FROM ordered_values
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Project classification dimension built from silver.dim_project. Includes UNKNOWN as surrogate key -1.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    duplicate_name_rows = spark.sql(f"""
        SELECT COUNT(*) AS dup_count
        FROM (
            SELECT
                  project_classification
                , COUNT(*) AS cnt
            FROM {TARGET_TABLE}
            GROUP BY project_classification
            HAVING COUNT(*) > 1
        ) d
    """).collect()[0]["dup_count"]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)
    print("Validation summary:")
    print(f"  total_rows           : {row_count:,}")
    print(f"  duplicate_name_rows  : {duplicate_name_rows:,}")

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise